# MMF1921 Project 1 Solution

This notebook solves Project 1 in Python only. It uses the original data files in `source/Python/`, keeps the instructor-provided `source/` folder unchanged, and writes result tables, figures, and the final Markdown report to `outputs/` and `MMF1921_Project_1_Final_Report.md`.

## Setup

The reusable implementation lives in `solution/`. The supplied CSV files remain in `source/Python/`.

In [1]:
from pathlib import Path
import sys

from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name != "project 1":
    PROJECT_ROOT = PROJECT_ROOT / "project 1"
if not (PROJECT_ROOT / "solution").exists():
    raise FileNotFoundError(
        "Run this notebook from the project 1 folder or its parent."
    )

SOLUTION_DIR = PROJECT_ROOT / "solution"
sys.path.insert(0, str(SOLUTION_DIR))

from project1_core import run_experiment, write_experiment_tables, write_figures
from reporting import (
    markdown_table,
    read_csv_rows,
    summarize_fit_by_model,
    write_report,
)

TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

## Run the Experiment

This cell calibrates OLS, Fama-French, LASSO, and BSS models over each four-year rolling window. It then runs long-only mean-variance optimization for each out-of-sample year from 2012 through 2016.

In [2]:
results = run_experiment()
write_experiment_tables(results)
write_figures(results)
report_path = write_report()

print(f"Wrote tables to {TABLE_DIR.relative_to(PROJECT_ROOT)}")
print(f"Wrote figures to {FIGURE_DIR.relative_to(PROJECT_ROOT)}")
print(f"Wrote report to {report_path.relative_to(PROJECT_ROOT)}")

Wrote tables to outputs/tables
Wrote figures to outputs/figures
Wrote report to MMF1921_Project_1_Final_Report.md


## In-Sample Fit Summary

Adjusted $R^2$ measures model fit while penalizing models that use more explanatory variables. Higher values indicate better in-sample explanatory power after that penalty.

In [3]:
_, fit_rows = read_csv_rows(TABLE_DIR / "in_sample_fit_summary.csv")
fit_summary_rows = summarize_fit_by_model(fit_rows)
display(
    Markdown(
        markdown_table(
            ["Model", "Mean adjusted R2", "Mean selected coefficients"],
            fit_summary_rows,
        )
    )
)

| Model | Mean adjusted R2 | Mean selected coefficients |
| --- | --- | --- |
| OLS | 0.4459 | 9.00 |
| FF | 0.3605 | 4.00 |
| LASSO | 0.3063 | 2.74 |
| BSS | 0.4660 | 4.00 |

## Out-of-Sample Portfolio Performance

Average return and volatility are monthly. Annualized return compounds the average monthly return over 12 months. Annualized volatility multiplies monthly volatility by $\sqrt{12}$.

In [4]:
performance_header, performance_rows = read_csv_rows(
    TABLE_DIR / "performance_summary.csv"
)
display(Markdown(markdown_table(performance_header, performance_rows)))

| model | average_monthly_return | monthly_volatility | annualized_return | annualized_volatility | sharpe_ratio | final_value |
| --- | --- | --- | --- | --- | --- | --- |
| OLS | 0.006760 | 0.026172 | 0.084202 | 0.090664 | 0.9287 | 146850.71 |
| FF | 0.006745 | 0.026251 | 0.084011 | 0.090935 | 0.9239 | 146699.53 |
| LASSO | 0.006768 | 0.026657 | 0.084303 | 0.092342 | 0.9129 | 146806.37 |
| BSS | 0.005538 | 0.026318 | 0.068518 | 0.091170 | 0.7515 | 136494.01 |

## Wealth Evolution

The figure below shows the monthly portfolio value paths for the four model-driven portfolios.

In [5]:
display(Markdown("![Portfolio wealth evolution](outputs/figures/wealth_evolution.svg)"))

![Portfolio wealth evolution](outputs/figures/wealth_evolution.svg)

## Portfolio Weights

The figure below shows BSS portfolio weights by rebalance period. BSS is a useful representative allocation plot because it is explicitly sparse.

In [6]:
display(Markdown("![BSS portfolio weights](outputs/figures/bss_weights.svg)"))

![BSS portfolio weights](outputs/figures/bss_weights.svg)

## Output Files

The full formal report is written to `MMF1921_Project_1_Final_Report.md`. Detailed CSV outputs are written under `outputs/tables/`, and SVG figures are written under `outputs/figures/`.

In [7]:
for path in sorted((PROJECT_ROOT / "outputs").glob("**/*")):
    if path.is_file():
        print(path.relative_to(PROJECT_ROOT))

outputs/figures/bss_weights.svg
outputs/figures/wealth_evolution.svg
outputs/tables/asset_fit_details.csv
outputs/tables/in_sample_fit_summary.csv
outputs/tables/performance_summary.csv
outputs/tables/portfolio_values.csv
outputs/tables/portfolio_weights.csv
